###Mount the drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


###Import necessary modules

In [ ]:
import csv
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

###Set Paths

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Data/hcc_msk_2024")
OUTPUT_DIR = Path("/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data")

DATA_DIR, OUTPUT_DIR


(PosixPath('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Data/hcc_msk_2024'),
 PosixPath('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data'))

###Helper Functions

In [ ]:
def load_cbio_clinical_table(path: Path) -> pd.DataFrame:
    with path.open() as handle:
        reader = csv.reader(handle, delimiter="\t")
        for row in reader:
            if row and not row[0].startswith("#"):
                header = row
                break
        else:
            raise ValueError(f"No header row found in {path.name}")

        rows = []
        for row in reader:
            if not row:
                continue
            if len(row) < len(header):
                row = row + [""] * (len(header) - len(row))
            rows.append(row[:len(header)])

    return pd.DataFrame(rows, columns=header)


def build_mutation_matrix(mutations_df: pd.DataFrame) -> pd.DataFrame:
    mutation_flags = (
        mutations_df[["Tumor_Sample_Barcode", "Hugo_Symbol"]]
        .dropna()
        .assign(MUTATED=1)
        .drop_duplicates()
        .pivot(index="Tumor_Sample_Barcode", columns="Hugo_Symbol", values="MUTATED")
        .fillna(0)
        .astype(np.int8)
    )
    mutation_flags.columns = [f"MUT_{gene}" for gene in mutation_flags.columns]
    return mutation_flags.reset_index().rename(columns={"Tumor_Sample_Barcode": "SAMPLE_ID"})


def build_cna_matrix(path: Path) -> pd.DataFrame:
    cna_df = pd.read_csv(path, sep="\t")
    cna_matrix = cna_df.set_index("Hugo_Symbol").transpose().reset_index()
    cna_matrix = cna_matrix.rename(columns={"index": "SAMPLE_ID"})
    cna_matrix.columns = ["SAMPLE_ID"] + [f"CNA_{gene}" for gene in cna_matrix.columns[1:]]
    return cna_matrix


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

### Load All files

In [ ]:

patient_df = load_cbio_clinical_table(DATA_DIR / "data_clinical_patient.txt")
sample_df = load_cbio_clinical_table(DATA_DIR / "data_clinical_sample.txt")
mutations_df = pd.read_csv(DATA_DIR / "data_mutations.txt", sep="\t", comment="#", low_memory=False)
cna_matrix = build_cna_matrix(DATA_DIR / "data_cna.txt")

patient_df.shape, sample_df.shape, mutations_df.shape, cna_matrix.shape

((1370, 26), (1370, 21), (6801, 124), (1370, 542))

Quick Preview

In [ ]:
patient_df.head()

,PATIENT_ID,SEX,RACE,ETHNICITY,DIAGNOSIS_AGE,OS_MONTHS,OS_STATUS,BMI,DIABETES,VIRAL_HEPATITIS,...,P_N,C_N,METASTASIS,CURATIVE_INTENT_SURGERY,MAJOR_LIVER_RESECTION,RESECTION_MARGIN,NEOADJUVANT_SYSTEMIC_THERAPY,ADJUVANT_SYSTEMIC_THERAPY,PALLIATIVE_SYSTEMIC_THERPAY,BILIARY_SCORE
0,P-0000037,Male,ASIAN,Unknown whether Spanish or not,,,0:LIVING,22.5,No,Yes,...,Negative,No,No,Yes,No,R0,No,No,No,
1,P-0000114,Female,WHITE,Non-Spanish; Non-Hispanic,57,15.84,1:DECEASED,21.4,No,No,...,Positive,No,No,Yes,Yes,R0,Yes,No,Yes,0.685
2,P-0000127,Male,WHITE,Non-Spanish; Non-Hispanic,62,15.048,1:DECEASED,19.5,No,Yes,...,,Yes,Yes,No,,,No,No,Yes,0.394
3,P-0000147,Female,WHITE,Non-Spanish; Non-Hispanic,40,72.48,1:DECEASED,23.1,No,No,...,Negative,No,No,Yes,No,R0,No,Yes,No,
4,P-0000154,Male,WHITE,Non-Spanish; Non-Hispanic,76,24.48,1:DECEASED,27.7,No,No,...,Negative,No,No,No,,,No,No,Yes,0.449


In [ ]:
sample_df.head()

,SAMPLE_ID,PATIENT_ID,CANCER_TYPE,SAMPLE_TYPE,SAMPLE_CLASS,METASTATIC_SITE,PRIMARY_SITE,CANCER_TYPE_DETAILED,GENE_PANEL,SAMPLE_COVERAGE,...,ONCOTREE_CODE,MSI_SCORE,MSI_TYPE,SOMATIC_STATUS,SUBGROUP,PRED_CATEGORY,HISTO_SUBTYPE,TUMOR_STAGE,TUMOR_SIZE,TMB_NONSYNONYMOUS
0,P-0000037-T02-IM3,P-0000037,Hepatobiliary Cancer,Primary,Tumor,Not Applicable,Liver,Hepatocellular Carcinoma,IMPACT341,201,...,HCC,0.27,Stable,Matched,HCC,,,Moderately differentiated,0.5,3.327465902
1,P-0000114-T01-IM3,P-0000114,Hepatobiliary Cancer,Metastasis,Tumor,Lymph Node,Liver,Intrahepatic Cholangiocarcinoma,IMPACT341,938,...,IHCH,0.47,Stable,Matched,IHC,Biliary-aligned,,Poorly differentiated,5,4.436621202
2,P-0000127-T01-IM3,P-0000127,Hepatobiliary Cancer,Metastasis,Tumor,Lymph Node,Liver,Intrahepatic Cholangiocarcinoma,IMPACT341,623,...,IHCH,0,Stable,Matched,IHC,HCC-aligned,,Poorly differentiated,11.1,2.218310601
3,P-0000147-T01-IM3,P-0000147,Hepatobiliary Cancer,Primary,Tumor,Not Applicable,Liver,Intrahepatic Cholangiocarcinoma,IMPACT341,1051,...,IHCH,0.17,Stable,Matched,IHC,,,Moderately differentiated,5,0
4,P-0000154-T01-IM3,P-0000154,Hepatobiliary Cancer,Primary,Tumor,Not Applicable,Liver,Intrahepatic Cholangiocarcinoma,IMPACT341,767,...,IHCH,1.2,Stable,Matched,IHC,HCC-aligned,,Moderately differentiated,9.2,3.327465902


In [ ]:
mutations_df[["Hugo_Symbol", "Variant_Classification", "Tumor_Sample_Barcode"]].head()

,Hugo_Symbol,Variant_Classification,Tumor_Sample_Barcode
0,TP53,Missense_Mutation,P-0028912-T01-IM6
1,FYN,Frame_Shift_Ins,P-0028912-T01-IM6
2,KDM5C,Missense_Mutation,P-0028912-T01-IM6
3,ATM,Frame_Shift_Del,P-0061390-T01-IM7
4,PRKD1,Missense_Mutation,P-0061390-T01-IM7


In [ ]:
cna_matrix.iloc[:5, :8]

,SAMPLE_ID,CNA_TAP1,CNA_ERRFI1,CNA_STK19,CNA_CRKL,CNA_SCG5,CNA_STK11,CNA_MEN1
0,P-0023512-T01-IM6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,P-0052287-T01-IM6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,P-0007205-T01-IM5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,P-0002211-T01-IM3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,P-0061290-T01-IM7,0.0,0.0,0.0,0.0,0.0,0.0,0.0


###Build survival label from patient clinical data

In [ ]:
patient_df["OS_MONTHS"] = pd.to_numeric(patient_df["OS_MONTHS"], errors="coerce")
patient_df["OS_EVENT"] = patient_df["OS_STATUS"].map({
    "1:DECEASED": 1,
    "0:LIVING": 0,
})

patient_df[["PATIENT_ID", "OS_MONTHS", "OS_STATUS", "OS_EVENT"]].head()

,PATIENT_ID,OS_MONTHS,OS_STATUS,OS_EVENT
0,P-0000037,NaN,0:LIVING,0.0
1,P-0000114,15.840,1:DECEASED,1.0
2,P-0000127,15.048,1:DECEASED,1.0
3,P-0000147,72.480,1:DECEASED,1.0
4,P-0000154,24.480,1:DECEASED,1.0


In [ ]:
print("Missing OS_MONTHS:", patient_df["OS_MONTHS"].isna().sum())
print("Missing OS_EVENT:", patient_df["OS_EVENT"].isna().sum())
print("OS_EVENT distribution:")
print(patient_df["OS_EVENT"].value_counts(dropna=False))

Missing OS_MONTHS: 13
Missing OS_EVENT: 3
OS_EVENT distribution:
OS_EVENT
1.0    807
0.0    560
NaN      3
Name: count, dtype: int64


###Check sample-patient mapping

In [ ]:
print("Unique patient IDs in patient file:", patient_df["PATIENT_ID"].nunique())
print("Unique patient IDs in sample file:", sample_df["PATIENT_ID"].nunique())
print("Unique sample IDs in sample file:", sample_df["SAMPLE_ID"].nunique())
print("Patients with multiple samples:", (sample_df["PATIENT_ID"].value_counts() > 1).sum())

Unique patient IDs in patient file: 1370
Unique patient IDs in sample file: 1370
Unique sample IDs in sample file: 1370
Patients with multiple samples: 0


###Prepare patient and sample clinical feature tables

In [ ]:
patient_feature_df = patient_df.drop(columns=["OS_STATUS"]).copy()
sample_feature_df = sample_df.copy()

numeric_candidate_cols = [
    "DIAGNOSIS_AGE",
    "BMI",
    "BILIARY_SCORE",
    "SAMPLE_COVERAGE",
    "TUMOR_PURITY",
    "MSI_SCORE",
    "TUMOR_SIZE",
    "TMB_NONSYNONYMOUS",
]

for col in numeric_candidate_cols:
    if col in patient_feature_df.columns:
        patient_feature_df[col] = pd.to_numeric(patient_feature_df[col], errors="coerce")
    if col in sample_feature_df.columns:
        sample_feature_df[col] = pd.to_numeric(sample_feature_df[col], errors="coerce")

patient_feature_df.head()

,PATIENT_ID,SEX,RACE,ETHNICITY,DIAGNOSIS_AGE,OS_MONTHS,BMI,DIABETES,VIRAL_HEPATITIS,STEATOSIS,...,C_N,METASTASIS,CURATIVE_INTENT_SURGERY,MAJOR_LIVER_RESECTION,RESECTION_MARGIN,NEOADJUVANT_SYSTEMIC_THERAPY,ADJUVANT_SYSTEMIC_THERAPY,PALLIATIVE_SYSTEMIC_THERPAY,BILIARY_SCORE,OS_EVENT
0,P-0000037,Male,ASIAN,Unknown whether Spanish or not,NaN,NaN,22.5,No,Yes,No,...,No,No,Yes,No,R0,No,No,No,NaN,0.0
1,P-0000114,Female,WHITE,Non-Spanish; Non-Hispanic,57.0,15.840,21.4,No,No,No,...,No,No,Yes,Yes,R0,Yes,No,Yes,0.685,1.0
2,P-0000127,Male,WHITE,Non-Spanish; Non-Hispanic,62.0,15.048,19.5,No,Yes,No,...,Yes,Yes,No,,,No,No,Yes,0.394,1.0
3,P-0000147,Female,WHITE,Non-Spanish; Non-Hispanic,40.0,72.480,23.1,No,No,No,...,No,No,Yes,No,R0,No,Yes,No,NaN,1.0
4,P-0000154,Male,WHITE,Non-Spanish; Non-Hispanic,76.0,24.480,27.7,No,No,No,...,No,No,No,,,No,No,Yes,0.449,1.0


##Build mutation feature matrix

In [ ]:
mutation_matrix = build_mutation_matrix(mutations_df)

mutation_matrix.shape



(1294, 487)

In [ ]:
mutation_matrix.head()

,SAMPLE_ID,MUT_ABL1,MUT_ACVR1,MUT_AGO2,MUT_AKT1,MUT_AKT2,MUT_AKT3,MUT_ALB,MUT_ALK,MUT_ALOX12B,...,MUT_WT1,MUT_WWTR1,MUT_XIAP,MUT_XPO1,MUT_XRCC2,MUT_YAP1,MUT_YES1,MUT_ZFHX3,MUT_ZNRF3,MUT_ZRSR2
0,P-0000037-T02-IM3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,P-0000114-T01-IM3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,P-0000127-T01-IM3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,P-0000154-T01-IM3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,P-0000159-T01-IM3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Merge all feature sources

In [ ]:
master_df = (
    sample_feature_df
    .merge(patient_feature_df, on="PATIENT_ID", how="inner", validate="one_to_one")
    .merge(mutation_matrix, on="SAMPLE_ID", how="left")
    .merge(cna_matrix, on="SAMPLE_ID", how="left", validate="one_to_one")
)

mutation_feature_cols = [col for col in mutation_matrix.columns if col != "SAMPLE_ID"]
master_df[mutation_feature_cols] = master_df[mutation_feature_cols].fillna(0)

master_df = master_df.dropna(subset=["OS_MONTHS", "OS_EVENT"]).copy()
master_df["OS_EVENT"] = master_df["OS_EVENT"].astype(int)

master_df.shape

(1357, 1073)

In [ ]:
master_df[["PATIENT_ID", "SAMPLE_ID", "OS_MONTHS", "OS_EVENT"]].head()

,PATIENT_ID,SAMPLE_ID,OS_MONTHS,OS_EVENT
1,P-0000114,P-0000114-T01-IM3,15.840,1
2,P-0000127,P-0000127-T01-IM3,15.048,1
3,P-0000147,P-0000147-T01-IM3,72.480,1
4,P-0000154,P-0000154-T01-IM3,24.480,1
5,P-0000159,P-0000159-T01-IM3,18.984,1


###Save the master merged table

In [ ]:
master_output_path = OUTPUT_DIR / "hcc_master_survival_features.csv"
master_df.to_csv(master_output_path, index=False)

master_output_path

PosixPath('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_master_survival_features.csv')

###Train-test split before fitting preprocessing

In [ ]:
id_cols = ["PATIENT_ID", "SAMPLE_ID"]
target_cols = ["OS_MONTHS", "OS_EVENT"]

train_idx, test_idx = train_test_split(
    master_df.index,
    test_size=0.2,
    random_state=42,
    stratify=master_df["OS_EVENT"],
)

train_raw = master_df.loc[train_idx].reset_index(drop=True)
test_raw = master_df.loc[test_idx].reset_index(drop=True)

train_raw.shape, test_raw.shape

((1085, 1073), (272, 1073))

In [ ]:
print("Train OS_EVENT distribution:")
print(train_raw["OS_EVENT"].value_counts())

print("\nTest OS_EVENT distribution:")
print(test_raw["OS_EVENT"].value_counts())

Train OS_EVENT distribution:
OS_EVENT
1    641
0    444
Name: count, dtype: int64

Test OS_EVENT distribution:
OS_EVENT
1    161
0    111
Name: count, dtype: int64


###Fit preprocessing on train only

In [ ]:
train_features = train_raw.drop(columns=id_cols + target_cols)
test_features = test_raw.drop(columns=id_cols + target_cols)

categorical_cols = train_features.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = [col for col in train_features.columns if col not in categorical_cols]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="median")),
            ]),
            numeric_cols,
        ),
        (
            "categorical",
            Pipeline(steps=[
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", make_one_hot_encoder()),
            ]),
            categorical_cols,
        ),
    ],
)

train_processed = preprocessor.fit_transform(train_features)
test_processed = preprocessor.transform(test_features)

len(numeric_cols), len(categorical_cols)

(1035, 34)

Build final train/test DataFrames

In [ ]:
categorical_feature_names = (
    preprocessor.named_transformers_["categorical"]["encoder"]
    .get_feature_names_out(categorical_cols)
    .tolist()
    if categorical_cols else []
)

processed_feature_names = numeric_cols + categorical_feature_names

train_processed_df = pd.DataFrame(train_processed, columns=processed_feature_names)
test_processed_df = pd.DataFrame(test_processed, columns=processed_feature_names)

train_final = pd.concat(
    [
        train_raw[id_cols].reset_index(drop=True),
        train_processed_df,
        train_raw[target_cols].reset_index(drop=True),
    ],
    axis=1,
)

test_final = pd.concat(
    [
        test_raw[id_cols].reset_index(drop=True),
        test_processed_df,
        test_raw[target_cols].reset_index(drop=True),
    ],
    axis=1,
)

train_final.shape, test_final.shape

((1085, 1196), (272, 1196))

In [ ]:
print("Missing values in train_final:", int(train_final.isna().sum().sum()))
print("Missing values in test_final:", int(test_final.isna().sum().sum()))

Missing values in train_final: 0
Missing values in test_final: 0


### Save final output

In [ ]:
train_output_path = OUTPUT_DIR / "hcc_train_full.csv"
test_output_path = OUTPUT_DIR / "hcc_test_full.csv"

train_final.to_csv(train_output_path, index=False)
test_final.to_csv(test_output_path, index=False)

train_output_path, test_output_path

(PosixPath('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_train_full.csv'),
 PosixPath('/content/drive/MyDrive/IEEE Project/Hepatocellular Carcinoma/Processed Data/hcc_test_full.csv'))

###Final Inspection

In [ ]:
train_final.head()

,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0011104,P-0011104-T01-IM5,886.0,60.0,0.00,6.4,5.872318,53.0,19.2,0.716,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,82.296,0
1,P-0001336,P-0001336-T01-IM3,755.0,30.0,0.16,6.4,4.436621,61.0,40.7,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,51.324,1
2,P-0064295,P-0064295-T01-IM7,635.0,70.0,0.50,5.4,4.101736,74.0,24.4,0.532,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,6.600,1
3,P-0049360,P-0049360-T01-IM6,537.0,30.0,0.12,6.4,4.323490,61.0,37.2,0.737,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,24.480,0
4,P-0010532,P-0010532-T01-IM5,1056.0,30.0,0.16,3.8,2.936159,63.0,20.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,14.592,1


In [ ]:
test_final.head()

,PATIENT_ID,SAMPLE_ID,SAMPLE_COVERAGE,TUMOR_PURITY,MSI_SCORE,TUMOR_SIZE,TMB_NONSYNONYMOUS,DIAGNOSIS_AGE,BMI,BILIARY_SCORE,...,NEOADJUVANT_SYSTEMIC_THERAPY_No,NEOADJUVANT_SYSTEMIC_THERAPY_Yes,ADJUVANT_SYSTEMIC_THERAPY_,ADJUVANT_SYSTEMIC_THERAPY_No,ADJUVANT_SYSTEMIC_THERAPY_Yes,PALLIATIVE_SYSTEMIC_THERPAY_,PALLIATIVE_SYSTEMIC_THERPAY_No,PALLIATIVE_SYSTEMIC_THERPAY_Yes,OS_MONTHS,OS_EVENT
0,P-0016909,P-0016909-T01-IM6,913.0,70.0,0.00,6.8,0.000000,59.0,24.4,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,28.980,1
1,P-0032234,P-0032234-T01-IM6,560.0,60.0,0.33,13.7,4.323490,76.0,26.0,0.951,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,5.952,1
2,P-0045819,P-0045819-T01-IM6,592.0,90.0,1.61,4.5,6.052887,59.0,29.9,0.763,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,9.984,0
3,P-0052368,P-0052368-T02-IM6,642.0,40.0,0.05,6.7,4.323490,63.0,27.7,0.868,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,23.652,1
4,P-0019203,P-0019203-T01-IM6,634.0,50.0,1.29,4.3,1.729396,57.0,21.6,0.763,...,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,36.696,0


####This notebook is the foundation of the whole survival analysis pipeline. Its role is to take the raw hcc_msk_2024 study files, convert them into a survival-ready machine learning dataset, and produce the first clean train-test split for downstream experiments.
The notebook begins by loading four study files:

data_clinical_patient.txt
data_clinical_sample.txt
data_mutations.txt
data_cna.txt

The patient clinical file provides the core survival outcome. From this file, we derive the survival target using:

OS_MONTHS as the survival duration

OS_STATUS converted into OS_EVENT, where 1:DECEASED -> 1 and 0:LIVING -> 0

This means the project is framed as an overall survival prediction problem, where the models will learn which patients are likely to experience death earlier versus later, based on their molecular and clinical characteristics.

The sample clinical file contributes tumor- and sample-level information such as:

SAMPLE_COVERAGE
TUMOR_PURITY
MSI_SCORE
TUMOR_SIZE
TMB_NONSYNONYMOUS

cancer subtype and other sample descriptors
The mutation file is transformed into a gene-level binary mutation matrix. Each row corresponds to a sample, and each gene is converted into a feature like:

MUT_TP53
MUT_ARID1A
MUT_CTNNB1

A value of 1 means the gene is mutated in that sample, and 0 means it is not.
The CNA file is transformed into a sample-by-gene copy number matrix. Each gene becomes a feature such as:

CNA_CDKN2A
CNA_KMT2B

This step allows the model to use gene-level copy number patterns together with mutation and clinical information.
After these separate feature blocks are prepared, the notebook merges them into one master table using:

PATIENT_ID
SAMPLE_ID

A key observation from this dataset is that the mapping is especially clean for Stage 1: there is effectively one sample per patient in the usable cohort, which makes the merge straightforward and avoids repeated-row complications.

Once the merged table is created, rows with missing survival labels are removed. This produces the final survival-ready cohort. At this stage, the notebook still keeps PATIENT_ID and SAMPLE_ID in the dataframe. These are not used as model input features, but they are intentionally retained for:

row tracking
linking predictions back to the right patient
future XAI interpretation
export tables and reporting

The notebook then performs preprocessing. Numeric variables are converted into proper numeric form, missing values are handled through imputation, and categorical variables are encoded using one-hot encoding.

Importantly, this fitting is done using the training set only, so no information leaks from the test set into preprocessing.
The final step is a stratified train-test split based on OS_EVENT, so that the event distribution remains reasonably balanced between train and test sets.
The notebook generates three main output files:

hcc_master_survival_features.csv
hcc_train_full.csv
hcc_test_full.csv

In practical terms, this notebook turns raw study files into a structured survival modeling dataset. It is the transition point from raw biomedical data to machine learning-ready survival data.